In [69]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import re
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    from src.feature_engineering import extract_resolution

In [70]:
from src.data_loader import load_data

from src.feature_engineering import extract_cpu_family

In [71]:
from src.feature_engineering import (
    extract_ssd,
    extract_hdd,
    extract_flash_storage,
    extract_hybrid,
    extract_resolution,
    extract_ips,
    extract_touchscreen,
)

In [72]:
from src.data_loader import load_data

DATA_PATH = Path("../data/processed/laptop_cleaned.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,company,product,type_name,inches,screen_resolution,cpu_company,cpu_type,cpu_frequency,ram,memory,gpu_company,gpu_type,operating_system,weight,price
0,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel,Core i5,2.3,8,128GB SSD,Intel,Iris Plus Graphics 640,macOS,1.37,1339.69
1,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel,Core i5,1.8,8,128GB Flash Storage,Intel,HD Graphics 6000,macOS,1.34,898.94
2,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel,Core i5 7200U,2.5,8,256GB SSD,Intel,HD Graphics 620,No OS,1.86,575.00
3,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel,Core i7,2.7,16,512GB SSD,AMD,Radeon Pro 455,macOS,1.83,2537.45
4,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel,Core i5,3.1,8,256GB SSD,Intel,Iris Plus Graphics 650,macOS,1.37,1803.60


In [73]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1275 entries, 0 to 1274
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   company            1275 non-null   str    
 1   product            1275 non-null   str    
 2   type_name          1275 non-null   str    
 3   inches             1275 non-null   float64
 4   screen_resolution  1275 non-null   str    
 5   cpu_company        1275 non-null   str    
 6   cpu_type           1275 non-null   str    
 7   cpu_frequency      1275 non-null   float64
 8   ram                1275 non-null   int64  
 9   memory             1275 non-null   str    
 10  gpu_company        1275 non-null   str    
 11  gpu_type           1275 non-null   str    
 12  operating_system   1275 non-null   str    
 13  weight             1275 non-null   float64
 14  price              1275 non-null   float64
dtypes: float64(4), int64(1), str(10)
memory usage: 149.5 KB


In [74]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
company,1275,19,Dell,291,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product,1275,618,XPS 13,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type_name,1275,6,Notebook,707,NaN,NaN,NaN,NaN,NaN,NaN,NaN
inches,1275.0,NaN,NaN,NaN,15.022902,1.42947,10.1,14.0,15.6,15.6,18.4
screen_resolution,1275,40,Full HD 1920x1080,505,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cpu_company,1275,3,Intel,1214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cpu_type,1275,93,Core i5 7200U,193,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cpu_frequency,1275.0,NaN,NaN,NaN,2.30298,0.503846,0.9,2.0,2.5,2.7,3.6
ram,1275.0,NaN,NaN,NaN,8.440784,5.097809,2.0,4.0,8.0,8.0,64.0
memory,1275,39,256GB SSD,412,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [75]:
feature_plan = pd.DataFrame({
    "Feature": df.columns,
    "Current Type": df.dtypes.astype(str).values,
    "Need Engineering": "",
    "New Features": "",
    "Notes": ""
})

feature_plan

,Feature,Current Type,Need Engineering,New Features,Notes
0,company,str,,,
1,product,str,,,
2,type_name,str,,,
3,inches,float64,,,
4,screen_resolution,str,,,
5,cpu_company,str,,,
6,cpu_type,str,,,
7,cpu_frequency,float64,,,
8,ram,int64,,,
9,memory,str,,,


In [76]:
feature_plan["Need Engineering"] = [
    "No",
    "Yes",
    "No",
    "No",
    "Yes",
    "No",
    "Yes",
    "No",
    "No",
    "Yes",
    "No",
    "Yes",
    "No",
    "No",
    "Target"
]

feature_plan["New Features"] = [
    "-",
    "Product Family",
    "-",
    "-",
    "Resolution, IPS, Touchscreen",
    "-",
    "CPU Family",
    "-",
    "-",
    "SSD, HDD, Flash, Hybrid",
    "-",
    "GPU Family",
    "-",
    "-",
    "-"
]

feature_plan["Notes"] = [
    "Encode later",
    "High Cardinality",
    "Encode later",
    "Keep",
    "Extract useful information",
    "Encode later",
    "Simplify CPU names",
    "Keep",
    "Keep",
    "Split memory information",
    "Encode later",
    "Simplify GPU names",
    "Encode later",
    "Keep",
    "Target variable"
]

feature_plan

,Feature,Current Type,Need Engineering,New Features,Notes
0,company,str,No,-,Encode later
1,product,str,Yes,Product Family,High Cardinality
2,type_name,str,No,-,Encode later
3,inches,float64,No,-,Keep
4,screen_resolution,str,Yes,"Resolution, IPS, Touchscreen",Extract useful information
5,cpu_company,str,No,-,Encode later
6,cpu_type,str,Yes,CPU Family,Simplify CPU names
7,cpu_frequency,float64,No,-,Keep
8,ram,int64,No,-,Keep
9,memory,str,Yes,"SSD, HDD, Flash, Hybrid",Split memory information


In [77]:
df["memory"].value_counts().head(20)

memory
256GB SSD               412
1TB HDD                 215
500GB HDD               124
512GB SSD               114
128GB SSD +  1TB HDD     94
128GB SSD                74
256GB SSD +  1TB HDD     73
32GB Flash Storage       36
2TB HDD                  16
512GB SSD +  1TB HDD     14
1TB SSD                  14
64GB Flash Storage       13
256GB SSD +  2TB HDD     10
256GB Flash Storage       8
1.0TB Hybrid              7
16GB Flash Storage        7
32GB SSD                  6
180GB SSD                 5
128GB Flash Storage       4
16GB SSD                  3
Name: count, dtype: int64

In [78]:
df["memory"].sample(10, random_state=42)

1179               500GB HDD
342                  1TB HDD
649                256GB SSD
772                128GB SSD
803                256GB SSD
358                  2TB HDD
44                   1TB HDD
231                500GB HDD
618     256GB SSD +  1TB HDD
43                 256GB SSD
Name: memory, dtype: str

## Memory Column Analysis

In [79]:
sorted(df["memory"].unique())

['1.0TB HDD',
 '1.0TB Hybrid',
 '128GB Flash Storage',
 '128GB HDD',
 '128GB SSD',
 '128GB SSD +  1TB HDD',
 '128GB SSD +  2TB HDD',
 '16GB Flash Storage',
 '16GB SSD',
 '180GB SSD',
 '1TB HDD',
 '1TB HDD +  1TB HDD',
 '1TB SSD',
 '1TB SSD +  1TB HDD',
 '240GB SSD',
 '256GB Flash Storage',
 '256GB SSD',
 '256GB SSD +  1.0TB Hybrid',
 '256GB SSD +  1TB HDD',
 '256GB SSD +  256GB SSD',
 '256GB SSD +  2TB HDD',
 '256GB SSD +  500GB HDD',
 '2TB HDD',
 '32GB Flash Storage',
 '32GB HDD',
 '32GB SSD',
 '500GB HDD',
 '508GB Hybrid',
 '512GB Flash Storage',
 '512GB SSD',
 '512GB SSD +  1.0TB Hybrid',
 '512GB SSD +  1TB HDD',
 '512GB SSD +  256GB SSD',
 '512GB SSD +  2TB HDD',
 '512GB SSD +  512GB SSD',
 '64GB Flash Storage',
 '64GB Flash Storage +  1TB HDD',
 '64GB SSD',
 '8GB SSD']

In [80]:
# Check for TB
df[df["memory"].str.contains("TB")]["memory"].unique()

<StringArray>
[                      '1TB HDD',          '128GB SSD +  1TB HDD',
          '256GB SSD +  1TB HDD',          '256GB SSD +  2TB HDD',
                       '2TB HDD',                  '1.0TB Hybrid',
          '512GB SSD +  1TB HDD',                       '1TB SSD',
          '128GB SSD +  2TB HDD',          '512GB SSD +  2TB HDD',
 '64GB Flash Storage +  1TB HDD',            '1TB HDD +  1TB HDD',
            '1TB SSD +  1TB HDD',                     '1.0TB HDD',
     '512GB SSD +  1.0TB Hybrid',     '256GB SSD +  1.0TB Hybrid']
Length: 16, dtype: str

In [81]:
# Checking for Flash Storage
df[df["memory"].str.contains("Flash")]["memory"].unique()

<StringArray>
[          '128GB Flash Storage',           '256GB Flash Storage',
            '32GB Flash Storage',            '64GB Flash Storage',
            '16GB Flash Storage', '64GB Flash Storage +  1TB HDD',
           '512GB Flash Storage']
Length: 7, dtype: str

In [82]:
# Check for Hybrid existence
df[df["memory"].str.contains("Hybrid")]["memory"].unique()

<StringArray>
[             '1.0TB Hybrid',              '508GB Hybrid',
 '512GB SSD +  1.0TB Hybrid', '256GB SSD +  1.0TB Hybrid']
Length: 4, dtype: str

In [83]:
# Checking for the presence of two memories
df[df["memory"].str.contains("\+")]["memory"].unique()

<StringArray>
[         '128GB SSD +  1TB HDD',        '256GB SSD +  256GB SSD',
          '256GB SSD +  1TB HDD',          '256GB SSD +  2TB HDD',
          '512GB SSD +  1TB HDD',        '256GB SSD +  500GB HDD',
          '128GB SSD +  2TB HDD',        '512GB SSD +  512GB SSD',
        '512GB SSD +  256GB SSD',          '512GB SSD +  2TB HDD',
 '64GB Flash Storage +  1TB HDD',            '1TB HDD +  1TB HDD',
            '1TB SSD +  1TB HDD',     '512GB SSD +  1.0TB Hybrid',
     '256GB SSD +  1.0TB Hybrid']
Length: 15, dtype: str

In [84]:
memory_examples = df["memory"].sample(10, random_state=42)

memory_examples

1179               500GB HDD
342                  1TB HDD
649                256GB SSD
772                128GB SSD
803                256GB SSD
358                  2TB HDD
44                   1TB HDD
231                500GB HDD
618     256GB SSD +  1TB HDD
43                 256GB SSD
Name: memory, dtype: str

In [85]:
for value in memory_examples:
    print(value)

500GB HDD
1TB HDD
256GB SSD
128GB SSD
256GB SSD
2TB HDD
1TB HDD
500GB HDD
256GB SSD +  1TB HDD
256GB SSD


In [86]:
for value in memory_examples:
    print(value.split("+"))

['500GB HDD']
['1TB HDD']
['256GB SSD']
['128GB SSD']
['256GB SSD']
['2TB HDD']
['1TB HDD']
['500GB HDD']
['256GB SSD ', '  1TB HDD']
['256GB SSD']


In [87]:
sample = "256GB SSD + 1TB HDD"

parts = sample.split("+")

parts

['256GB SSD ', ' 1TB HDD']

In [88]:
for part in parts:
    print(part.strip())

256GB SSD
1TB HDD


In [89]:
sample = "256GB SSD + 1TB HDD"

parts = sample.split("+")
parts = [part.strip() for part in parts]

parts

['256GB SSD', '1TB HDD']

In [90]:
for part in parts:
    capacity = part.split()[0]
    print(capacity)

256GB
1TB


In [91]:
for part in parts:

    capacity = part.split()[0]

    if "TB" in capacity:
        capacity = float(capacity.replace("TB", "")) * 1024

    else:
        capacity = float(capacity.replace("GB", ""))

    print(capacity)

256.0
1024.0


In [92]:
extract_ssd("64GB Flash Storage")

0

In [93]:
df["ssd"] = df["memory"].apply(extract_ssd)

In [94]:
df[["memory", "ssd"]].head(10)

,memory,ssd
0,128GB SSD,128.0
1,128GB Flash Storage,0.0
2,256GB SSD,256.0
3,512GB SSD,512.0
4,256GB SSD,256.0
5,500GB HDD,0.0
6,256GB Flash Storage,0.0
7,256GB Flash Storage,0.0
8,512GB SSD,512.0
9,256GB SSD,256.0


In [95]:
(df["ssd"] > 0).sum()
# Count the number of SSDs

np.int64(837)

In [96]:
df[df["ssd"] > 0][["memory", "ssd"]].sample(10, random_state=42)

,memory,ssd
312,256GB SSD + 1TB HDD,256.0
1243,512GB SSD,512.0
119,256GB SSD,256.0
105,256GB SSD,256.0
600,256GB SSD,256.0
572,256GB SSD,256.0
55,256GB SSD,256.0
427,256GB SSD,256.0
679,256GB SSD,256.0
533,32GB SSD,32.0


In [97]:
df["ssd"].max()
# Maximum SSD

np.float64(1024.0)

In [98]:
sorted(df["ssd"].unique())

[np.float64(0.0),
 np.float64(8.0),
 np.float64(16.0),
 np.float64(32.0),
 np.float64(64.0),
 np.float64(128.0),
 np.float64(180.0),
 np.float64(240.0),
 np.float64(256.0),
 np.float64(512.0),
 np.float64(768.0),
 np.float64(1024.0)]

In [99]:
df[df["ssd"] == 180][["memory", "ssd"]]

,memory,ssd
477,180GB SSD,180.0
495,180GB SSD,180.0
753,180GB SSD,180.0
873,180GB SSD,180.0
1087,180GB SSD,180.0


In [100]:
df[df["ssd"] == 240][["memory", "ssd"]]

,memory,ssd
911,240GB SSD,240.0


Test

In [101]:

from src.feature_engineering import (
    extract_capacity,
    extract_ssd,
    extract_hdd,
    extract_flash_storage,
    extract_hybrid
)

In [102]:
extract_ssd("512GB SSD + 512GB SSD")

1024.0

In [103]:
extract_hdd("512GB SSD + 1TB HDD")

1024.0

In [104]:
extract_flash_storage("64GB Flash Storage")

64.0

In [105]:
extract_hybrid("1.0TB Hybrid")

1024.0

In [106]:
df["ssd"] = df["memory"].apply(extract_ssd)

df["hdd"] = df["memory"].apply(extract_hdd)

df["flash_storage"] = df["memory"].apply(extract_flash_storage)

df["hybrid"] = df["memory"].apply(extract_hybrid)

In [107]:
df.head()

,company,product,type_name,inches,screen_resolution,cpu_company,cpu_type,cpu_frequency,ram,memory,gpu_company,gpu_type,operating_system,weight,price,ssd,hdd,flash_storage,hybrid
0,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel,Core i5,2.3,8,128GB SSD,Intel,Iris Plus Graphics 640,macOS,1.37,1339.69,128.0,0.0,0.0,0.0
1,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel,Core i5,1.8,8,128GB Flash Storage,Intel,HD Graphics 6000,macOS,1.34,898.94,0.0,0.0,128.0,0.0
2,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel,Core i5 7200U,2.5,8,256GB SSD,Intel,HD Graphics 620,No OS,1.86,575.00,256.0,0.0,0.0,0.0
3,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel,Core i7,2.7,16,512GB SSD,AMD,Radeon Pro 455,macOS,1.83,2537.45,512.0,0.0,0.0,0.0
4,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel,Core i5,3.1,8,256GB SSD,Intel,Iris Plus Graphics 650,macOS,1.37,1803.60,256.0,0.0,0.0,0.0


Save new dataset

In [108]:
from pathlib import Path

FEATURED_DATA_PATH = Path("../data/processed/laptop_featured.csv")

df.to_csv(FEATURED_DATA_PATH, index=False)

print("Feature engineered dataset saved successfully.")

Feature engineered dataset saved successfully.


In [109]:
# Test the saved file
df_featured = pd.read_csv("../data/processed/laptop_featured.csv")

df_featured.head()

,company,product,type_name,inches,screen_resolution,cpu_company,cpu_type,cpu_frequency,ram,memory,gpu_company,gpu_type,operating_system,weight,price,ssd,hdd,flash_storage,hybrid
0,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel,Core i5,2.3,8,128GB SSD,Intel,Iris Plus Graphics 640,macOS,1.37,1339.69,128.0,0.0,0.0,0.0
1,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel,Core i5,1.8,8,128GB Flash Storage,Intel,HD Graphics 6000,macOS,1.34,898.94,0.0,0.0,128.0,0.0
2,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel,Core i5 7200U,2.5,8,256GB SSD,Intel,HD Graphics 620,No OS,1.86,575.00,256.0,0.0,0.0,0.0
3,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel,Core i7,2.7,16,512GB SSD,AMD,Radeon Pro 455,macOS,1.83,2537.45,512.0,0.0,0.0,0.0
4,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel,Core i5,3.1,8,256GB SSD,Intel,Iris Plus Graphics 650,macOS,1.37,1803.60,256.0,0.0,0.0,0.0


# Summary

In this notebook, we performed the first stage of feature engineering.

Completed tasks:

- Extracted SSD capacity
- Extracted HDD capacity
- Extracted Flash Storage capacity
- Extracted Hybrid Storage capacity
- Validated generated features
- Saved the processed dataset

Output:

data/processed/laptop_featured.csv

/////////////////////////////////////////////////////////////////////

Now we move on to the next feature.

In [110]:
df["screen_resolution"].head(15)

0     IPS Panel Retina Display 2560x1600
1                               1440x900
2                      Full HD 1920x1080
3     IPS Panel Retina Display 2880x1800
4     IPS Panel Retina Display 2560x1600
5                               1366x768
6     IPS Panel Retina Display 2880x1800
7                               1440x900
8                      Full HD 1920x1080
9            IPS Panel Full HD 1920x1080
10                              1366x768
11                     Full HD 1920x1080
12    IPS Panel Retina Display 2880x1800
13                     Full HD 1920x1080
14    IPS Panel Retina Display 2304x1440
Name: screen_resolution, dtype: str

In [111]:
df["screen_resolution"].sample(15, random_state=42)

1179                               1366x768
342             IPS Panel Full HD 1920x1080
649             IPS Panel Full HD 1920x1080
772                      IPS Panel 1366x768
803     4K Ultra HD / Touchscreen 3840x2160
358                    Touchscreen 1366x768
44          Full HD / Touchscreen 1920x1080
231                                1366x768
618                       Full HD 1920x1080
43              IPS Panel Full HD 1920x1080
588                    Touchscreen 1366x768
63                        Full HD 1920x1080
1015                               1366x768
270      IPS Panel Retina Display 2560x1600
1140        IPS Panel Touchscreen 2560x1440
Name: screen_resolution, dtype: str

In [112]:
df["screen_resolution"].nunique()

40

In [113]:
for value in sorted(df["screen_resolution"].unique()):
    print(value)

1366x768
1440x900
1600x900
1920x1080
2560x1440
4K Ultra HD / Touchscreen 3840x2160
4K Ultra HD 3840x2160
Full HD / Touchscreen 1920x1080
Full HD 1920x1080
IPS Panel 1366x768
IPS Panel 2560x1440
IPS Panel 4K Ultra HD / Touchscreen 3840x2160
IPS Panel 4K Ultra HD 3840x2160
IPS Panel Full HD / Touchscreen 1920x1080
IPS Panel Full HD 1366x768
IPS Panel Full HD 1920x1080
IPS Panel Full HD 1920x1200
IPS Panel Full HD 2160x1440
IPS Panel Full HD 2560x1440
IPS Panel Quad HD+ / Touchscreen 3200x1800
IPS Panel Quad HD+ 2560x1440
IPS Panel Quad HD+ 3200x1800
IPS Panel Retina Display 2304x1440
IPS Panel Retina Display 2560x1600
IPS Panel Retina Display 2736x1824
IPS Panel Retina Display 2880x1800
IPS Panel Touchscreen / 4K Ultra HD 3840x2160
IPS Panel Touchscreen 1366x768
IPS Panel Touchscreen 1920x1200
IPS Panel Touchscreen 2400x1600
IPS Panel Touchscreen 2560x1440
Quad HD+ / Touchscreen 3200x1800
Quad HD+ 3200x1800
Touchscreen / 4K Ultra HD 3840x2160
Touchscreen / Full HD 1920x1080
Touchscreen /

In [114]:
text = "IPS Panel Full HD 1920x1080"

re.findall(r"\d+x\d+", text)

['1920x1080']

In [115]:
tests = [
    "1366x768",
    "Touchscreen 2256x1504",
    "IPS Panel Retina Display 2560x1600",
    "4K Ultra HD 3840x2160"
]

for text in tests:
    print(re.findall(r"\d+x\d+", text))

['1366x768']
['2256x1504']
['2560x1600']
['3840x2160']


In [116]:
extract_resolution("IPS Panel Full HD 1920x1080")

(1920, 1080)

In [117]:
# Applying a function to a DataFrame
df[["resolution_width", "resolution_height"]] = df["screen_resolution"].apply(
    lambda x: pd.Series(extract_resolution(x))
)

In [118]:
df[
    [
        "screen_resolution",
        "resolution_width",
        "resolution_height"
    ]
].head(15)

,screen_resolution,resolution_width,resolution_height
0,IPS Panel Retina Display 2560x1600,2560,1600
1,1440x900,1440,900
2,Full HD 1920x1080,1920,1080
3,IPS Panel Retina Display 2880x1800,2880,1800
4,IPS Panel Retina Display 2560x1600,2560,1600
5,1366x768,1366,768
6,IPS Panel Retina Display 2880x1800,2880,1800
7,1440x900,1440,900
8,Full HD 1920x1080,1920,1080
9,IPS Panel Full HD 1920x1080,1920,1080


In [119]:
df["resolution_width"].describe()

count    1275.000000
mean     1900.043922
std       493.346186
min      1366.000000
25%      1920.000000
50%      1920.000000
75%      1920.000000
max      3840.000000
Name: resolution_width, dtype: float64

In [120]:
df["resolution_height"].describe()

count    1275.000000
mean     1073.904314
std       283.883940
min       768.000000
25%      1080.000000
50%      1080.000000
75%      1080.000000
max      2160.000000
Name: resolution_height, dtype: float64

In [121]:
sorted(df["resolution_width"].unique())

[np.int64(1366),
 np.int64(1440),
 np.int64(1600),
 np.int64(1920),
 np.int64(2160),
 np.int64(2256),
 np.int64(2304),
 np.int64(2400),
 np.int64(2560),
 np.int64(2736),
 np.int64(2880),
 np.int64(3200),
 np.int64(3840)]

In [122]:
sorted(df["resolution_height"].unique())

[np.int64(768),
 np.int64(900),
 np.int64(1080),
 np.int64(1200),
 np.int64(1440),
 np.int64(1504),
 np.int64(1600),
 np.int64(1800),
 np.int64(1824),
 np.int64(2160)]

In [123]:
# Test
extract_resolution("IPS Panel Full HD 1920x1080")
# This means that Notebook is no longer using the code in src.

(1920, 1080)

In [124]:
"IPS Panel" in "IPS Panel Full HD 1920x1080"

True

In [125]:
"IPS Panel" in "1366x768"

False

Now we will create the next feature.

extract_ips function

In [126]:
extract_ips("IPS Panel Full HD 1920x1080")

1

In [127]:
extract_ips("1366x768")

0

In [128]:
df["ips_panel"] = df["screen_resolution"].apply(extract_ips)

In [129]:
df[
    [
        "screen_resolution",
        "ips_panel"
    ]
].head(20)

,screen_resolution,ips_panel
0,IPS Panel Retina Display 2560x1600,1
1,1440x900,0
2,Full HD 1920x1080,0
3,IPS Panel Retina Display 2880x1800,1
4,IPS Panel Retina Display 2560x1600,1
5,1366x768,0
6,IPS Panel Retina Display 2880x1800,1
7,1440x900,0
8,Full HD 1920x1080,0
9,IPS Panel Full HD 1920x1080,1


In [130]:
df["ips_panel"].value_counts()

ips_panel
0    918
1    357
Name: count, dtype: int64

In [131]:
extract_touchscreen("Full HD / Touchscreen 1920x1080")

1

In [132]:
extract_touchscreen("IPS Panel Full HD 1920x1080")

0

In [133]:
extract_touchscreen("IPS Panel Touchscreen 2560x1440")

1

In [134]:
df["touchscreen"] = df["screen_resolution"].apply(extract_touchscreen)

In [135]:
df[
    [
        "screen_resolution",
        "touchscreen"
    ]
].head(20)

,screen_resolution,touchscreen
0,IPS Panel Retina Display 2560x1600,0
1,1440x900,0
2,Full HD 1920x1080,0
3,IPS Panel Retina Display 2880x1800,0
4,IPS Panel Retina Display 2560x1600,0
5,1366x768,0
6,IPS Panel Retina Display 2880x1800,0
7,1440x900,0
8,Full HD 1920x1080,0
9,IPS Panel Full HD 1920x1080,0


In [136]:
df["touchscreen"].value_counts()

touchscreen
0    1087
1     188
Name: count, dtype: int64

In [137]:
df["touchscreen"].value_counts(normalize=True)

touchscreen
0    0.852549
1    0.147451
Name: proportion, dtype: float64

In [138]:
extract_ssd("256GB SSD")

256.0

In [139]:
extract_resolution("Full HD 1920x1080")

(1920, 1080)

In [140]:
FEATURED_DATA_PATH = Path("../data/processed/laptop_featured.csv")

df.to_csv(FEATURED_DATA_PATH, index=False)

print("Featured dataset saved successfully!")
# Save new dataset

Featured dataset saved successfully!


////////////////////////////////////

In [141]:
df["cpu_type"].sample(30, random_state=42)

1179        Core i3 6100U
342         Core i3 7100U
649         Core i7 7500U
772         Core i5 6200U
803        Core i5 7300HQ
358         Core i5 7200U
44          Core i5 8250U
231        E-Series 9000e
618        Core i7 7700HQ
43          Core i5 8250U
588         Core i7 8550U
63          Core i5 8250U
1015        Core i5 6200U
270               Core i5
1140        Core i7 6600U
76          Core i5 7200U
81                Core i5
643        Core i7 7700HQ
670         Core i7 7600U
486        Core i7 6820HQ
155         Core i5 8250U
718            Atom Z8350
867         Core i7 7500U
771         Core i7 7500U
1181        Core i7 7500U
781        Core i7 6700HQ
1030        Core i5 7200U
858         Core i5 6200U
732     A9-Series A9-9420
294         Core i5 8250U
Name: cpu_type, dtype: str

In [142]:
extract_cpu_family("Core i7 7700HQ")

'Core i7'

In [143]:
extract_cpu_family("Atom Z8350")

'Atom'

In [144]:
extract_cpu_family("A9-Series A9-9420")

'AMD'

In [145]:
df["cpu_family"] = df["cpu_type"].apply(extract_cpu_family)

In [146]:
df["cpu_family"].value_counts()

cpu_family
Core i7    515
Core i5    423
Core i3    134
Celeron     78
Other       63
Pentium     30
AMD         19
Atom        13
Name: count, dtype: int64

In [147]:
df["cpu_family"].unique()

<StringArray>
['Core i5', 'Core i7', 'AMD', 'Core i3', 'Other', 'Atom', 'Celeron',
 'Pentium']
Length: 8, dtype: str

In [148]:
df["cpu_family"].nunique()

8

In [149]:
df["cpu_family"] = df["cpu_type"].apply(extract_cpu_family)

In [150]:
FEATURED_DATA_PATH = Path("../data/processed/laptop_featured.csv")

df.to_csv(FEATURED_DATA_PATH, index=False)

print("Featured dataset updated successfully!")

Featured dataset updated successfully!
